# AI Personal Tutor — Simple Complete Kaggle Version

### Input
- Book chapter
- Lecture notes

### RAG
- Document loading
- Chunking
- SentenceTransformer embeddings
- FAISS retrieval
- Retrieved context sent to Mistral

### Chains
1. Explain Topics
2. Generate Questions
3. Evaluate Answers

### Memory
Simple student memory:
- Quiz attempts
- Previous scores
- Last score

### Output Parser
Pydantic models validate the structured JSON produced by Mistral.

### Output
- Topic
- Explanation
- Example
- Quiz
- Student Score
- Feedback

In [1]:
!pip -q install transformers==4.52.4 accelerate bitsandbytes sentence-transformers faiss-cpu pypdf fastapi uvicorn pyngrok streamlit requests python-multipart pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 84.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 82.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.0 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is i

In [2]:
import json
import re
import secrets
import threading
import time
from pathlib import Path
from io import BytesIO

import torch
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from fastapi import FastAPI, UploadFile, File, Header, HTTPException
from pydantic import BaseModel
from kaggle_secrets import UserSecretsClient

In [6]:
secrets_client = UserSecretsClient()

NGROK_TOKEN = secrets_client.get_secret("NGROK_AUTH_TOKEN")
API_KEY = secrets_client.get_secret("TUTOR_API_KEY")
HF_TOKEN = secrets_client.get_secret("HF_TOKEN")

MODEL_ID = "mistralai/Mistral-Nemo-Instruct-2407"
EMBED_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
PORT = 8000
MAX_FILE_MB = 50

## 1. Load Mistral Nemo and the embedding model

Mistral is the tutor model.

SentenceTransformer creates vectors for the uploaded material and student questions.

In [7]:
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    use_fast=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=dtype,
)

model.eval()

embedder = SentenceTransformer(
    EMBED_MODEL_ID,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 2. Mistral generation

In [8]:
def generate(prompt, max_new_tokens=700):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 3. RAG: read, split, embed and retrieve

The uploaded chapter/lecture becomes the tutor's knowledge base.

The important RAG steps are:

**Document → Chunks → Embeddings → FAISS → Relevant Chunks**

In [9]:
lecture_chunks = []
lecture_index = None
lecture_name = ""

student_memory = {
    "quiz_attempts": [],
    "last_score": None,
}

current_quiz = None


def read_material(filename, data):
    extension = Path(filename).suffix.lower()

    if extension == ".pdf":
        reader = PdfReader(BytesIO(data))
        pages = []

        for page_number, page in enumerate(reader.pages, 1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append((page_number, text))

        return pages

    if extension in [".txt", ".md"]:
        text = data.decode("utf-8", errors="ignore")
        return [(1, text)]

    raise ValueError("Only PDF, TXT and MD are supported.")


def split_text(text, size=900, overlap=150):
    text = re.sub(r"\s+", " ", text).strip()

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


def index_material(filename, data):
    global lecture_chunks
    global lecture_index
    global lecture_name
    global student_memory

    pages = read_material(filename, data)
    chunks = []

    for page, text in pages:
        for chunk in split_text(text):
            if chunk:
                chunks.append({"page": page, "text": chunk})

    if not chunks:
        raise ValueError("No readable text found.")

    embeddings = embedder.encode(
        [x["text"] for x in chunks],
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    lecture_index = faiss.IndexFlatIP(embeddings.shape[1])
    lecture_index.add(embeddings)

    lecture_chunks = chunks
    lecture_name = filename

    # New lecture = new student learning session.
    student_memory = {
        "quiz_attempts": [],
        "last_score": None,
    }

    return {"filename": filename, "chunks": len(chunks)}


def retrieve(question, k=5):
    if lecture_index is None:
        raise ValueError("Upload material first.")

    query = embedder.encode(
        [question],
        normalize_embeddings=True,
    ).astype("float32")

    scores, indexes = lecture_index.search(
        query,
        min(k, len(lecture_chunks)),
    )

    return [
        {
            "page": lecture_chunks[i]["page"],
            "text": lecture_chunks[i]["text"],
            "score": float(score),
        }
        for score, i in zip(scores[0], indexes[0])
    ]


def context(results):
    return "\n\n".join(
        f"[Page {x['page']}]\n{x['text']}"
        for x in results
    )

## 4. Output Parser

Mistral returns structured JSON.

Pydantic validates that JSON before the application uses it.

This is the project's **Output Parser**.

In [10]:
class ExplanationOutput(BaseModel):
    topic: str
    explanation: str
    example: str


class QuizQuestion(BaseModel):
    id: int
    question: str
    options: list[str]
    answer: str
    explanation: str


class QuizOutput(BaseModel):
    questions: list[QuizQuestion]


def parse_json(text):
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object returned by the model.")

    return json.loads(text[start:end + 1])

## 5. Chain 1 — Explain Topics

RAG retrieves the lecture content.

Mistral creates:

- Topic
- Explanation
- Example

Pydantic parses the result.

In [11]:
def explain_topic():
    results = retrieve(
        "main topic important concepts definitions explanation example",
        k=6,
    )

    prompt = f'''
You are a personal tutor.

Use ONLY the retrieved material.

Return ONLY valid JSON:
{{
  "topic": "main topic",
  "explanation": "clear explanation for a student",
  "example": "simple example supported by the material"
}}

Retrieved material:
{context(results)}
'''

    parsed = ExplanationOutput.model_validate(
        parse_json(generate(prompt))
    )

    return parsed.model_dump()

## 6. Chain 2 — Generate Questions

RAG retrieves important lecture content.

Mistral creates the quiz.

Pydantic validates the quiz structure.

In [12]:
def generate_quiz(number):
    results = retrieve(
        "important concepts definitions examples and key ideas",
        k=8,
    )

    prompt = f'''
Create exactly {number} multiple-choice questions.

Use ONLY the retrieved material.

Return ONLY valid JSON:
{{
  "questions": [
    {{
      "id": 1,
      "question": "question",
      "options": ["A", "B", "C", "D"],
      "answer": "correct option",
      "explanation": "why it is correct"
    }}
  ]
}}

Retrieved material:
{context(results)}
'''

    parsed = QuizOutput.model_validate(
        parse_json(generate(prompt, 1400))
    )

    if len(parsed.questions) != number:
        raise ValueError("Wrong number of generated questions.")

    quiz_id = secrets.token_hex(4)

    global current_quiz
    current_quiz = {
        "id": quiz_id,
        "questions": parsed.questions,
    }

    return {
        "quiz_id": quiz_id,
        "questions": [
            {
                "id": q.id,
                "question": q.question,
                "options": q.options,
            }
            for q in parsed.questions
        ],
    }

## 7. Chain 3 — Evaluate Answers

The student's answers are compared with the parsed quiz.

The system produces:

- Student Score
- Percentage
- Feedback

The result is saved into **Memory**.

In [13]:
def evaluate_answers(quiz_id, answers):
    if current_quiz is None or current_quiz["id"] != quiz_id:
        raise ValueError("Invalid quiz.")

    results = []

    for q in current_quiz["questions"]:
        student = answers.get(str(q.id), "")
        correct = student.strip().lower() == q.answer.strip().lower()

        results.append({
            "id": q.id,
            "question": q.question,
            "student_answer": student,
            "correct_answer": q.answer,
            "score": 1 if correct else 0,
            "feedback": "Correct!" if correct else q.explanation,
        })

    score = sum(x["score"] for x in results)
    maximum = len(results)
    percent = round(score * 100 / maximum)

    if percent == 100:
        feedback = "Excellent. You answered everything correctly."
    elif percent >= 70:
        feedback = "Good work. Review the questions you missed."
    elif percent >= 50:
        feedback = "You understand some of the material. Review the missed concepts."
    else:
        feedback = "Review the material and try the quiz again."

    # MEMORY
    student_memory["last_score"] = percent
    student_memory["quiz_attempts"].append({
        "score": score,
        "max_score": maximum,
        "percent": percent,
    })

    return {
        "score": score,
        "max_score": maximum,
        "percent": percent,
        "overall_feedback": feedback,
        "results": results,
        "attempts": student_memory["quiz_attempts"],
    }

## 8. RAG Question Answering + Memory

The student can ask questions about the uploaded material.

The tutor retrieves the relevant chunks and answers from them.

The tutor also has access to the student's simple learning memory:
previous quiz attempts and scores.

In [14]:
def ask_tutor(question):
    results = retrieve(question, k=5)

    memory_text = ""

    if student_memory["last_score"] is not None:
        memory_text = (
            f"The student's last quiz score was "
            f"{student_memory['last_score']}%."
        )

    prompt = f'''
You are a personal tutor.

Answer using ONLY the retrieved material.

Do not use outside knowledge.

If the answer is not in the retrieved material, say:
"The uploaded material does not contain enough information."

{memory_text}

Student question:
{question}

Retrieved material:
{context(results)}
'''

    return {"answer": generate(prompt, 500)}

## 9. FastAPI

In [15]:
app = FastAPI(title="AI Personal Tutor")


def check_key(x_api_key):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid API key")


class QuizRequest(BaseModel):
    number: int = 5


class EvaluateRequest(BaseModel):
    quiz_id: str
    answers: dict


class AskRequest(BaseModel):
    question: str


@app.get("/health")
def health(x_api_key: str = Header(default="")):
    check_key(x_api_key)
    return {"status": "ok", "material": lecture_name}


@app.post("/upload")
def upload(file: UploadFile = File(...), x_api_key: str = Header(default="")):
    check_key(x_api_key)
    data = file.file.read()

    if len(data) > MAX_FILE_MB * 1024 * 1024:
        raise HTTPException(status_code=413, detail="File is too large.")

    try:
        return index_material(file.filename, data)
    except ValueError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Indexing failed: {type(e).__name__}: {e}")


@app.post("/explain")
def explain(x_api_key: str = Header(default="")):
    check_key(x_api_key)
    return explain_topic()


@app.post("/quiz")
def quiz(request: QuizRequest, x_api_key: str = Header(default="")):
    check_key(x_api_key)

    if request.number < 1 or request.number > 10:
        raise HTTPException(status_code=400, detail="Choose 1 to 10 questions.")

    return generate_quiz(request.number)


@app.post("/evaluate")
def evaluate(request: EvaluateRequest, x_api_key: str = Header(default="")):
    check_key(x_api_key)
    return evaluate_answers(request.quiz_id, request.answers)


@app.post("/ask")
def ask(request: AskRequest, x_api_key: str = Header(default="")):
    check_key(x_api_key)
    return ask_tutor(request.question)


@app.get("/memory")
def memory(x_api_key: str = Header(default="")):
    check_key(x_api_key)

    return {
        "attempts": len(student_memory["quiz_attempts"]),
        "last_score": student_memory["last_score"],
        "scores": [
            x["percent"]
            for x in student_memory["quiz_attempts"]
        ],
    }

## 10. Start FastAPI + ngrok

In [16]:
import uvicorn

server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="127.0.0.1",
        port=PORT,
        log_level="warning",
    )
)

thread = threading.Thread(
    target=server.run,
    daemon=True,
)

thread.start()

while not server.started:
    time.sleep(0.1)

# Streamlit runs in the same Kaggle machine, so it can call FastAPI directly.
# Only Streamlit needs ngrok.
print("Tutor API running at http://127.0.0.1:8000")
print("API key is loaded from the Kaggle secret and will not be entered in Streamlit.")


Tutor API running at http://127.0.0.1:8000
API key is loaded from the Kaggle secret and will not be entered in Streamlit.


## 11. Start Streamlit

In [17]:
# Upload app.py to a Kaggle Dataset and attach it to this notebook.
APP_PATH = "/kaggle/input/datasets/ahmedsuma/ai-personal/app (1).py"

print("App path:", APP_PATH)

App path: /kaggle/input/datasets/ahmedsuma/ai-personal/app (1).py


In [22]:
import os
import subprocess
import time
from pyngrok import ngrok

env = os.environ.copy()
env["TUTOR_API_KEY"] = API_KEY
env["TUTOR_BACKEND_URL"] = "http://127.0.0.1:8000"
ngrok.set_auth_token(NGROK_TOKEN)

streamlit_process = subprocess.Popen([
    "streamlit",
    "run",
    APP_PATH,
    "--server.port",
    "8501",
    "--server.address",
    "0.0.0.0",
    "--server.headless",
    "true",
], env=env)

time.sleep(3)

streamlit_tunnel = ngrok.connect(8501)

print("Streamlit URL:", streamlit_tunnel.public_url)
print("Backend: connected internally (no URL/API key entry needed)")

2026-10-07 19:46:15.762 Port 8501 is not available


Streamlit URL: https://duration-hanky-nutlike.ngrok-free.dev
Backend: connected internally (no URL/API key entry needed)
